# Harry Winston 価格スクレイパー
指定URLから商品名・価格・商品詳細を取得する

In [19]:
# Install dependencies if needed
# !pip install playwright beautifulsoup4 pandas
# !playwright install chromium


In [24]:
import re
import time
import asyncio
import pandas as pd
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright


In [25]:
URL = (
    "https://www.harrywinston.com/ja/products/sparkling-cluster-by-harry-winston/"
    "sparkling-cluster-sapphire-aquamarine-and-diamond-ring-frsaqpclrfspc"
)


In [45]:
_USER_AGENT = (
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/124.0.0.0 Safari/537.36"
)

_CONTEXT_OPTIONS = dict(
    user_agent=_USER_AGENT,
    locale="ja-JP",
    extra_http_headers={"Accept-Language": "ja,en-US;q=0.9,en;q=0.8"},
)


async def fetch_page(url: str, timeout_ms: int = 60_000) -> BeautifulSoup:
    """Fetch URL via headless Chromium (async Playwright) and return BeautifulSoup."""
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(**_CONTEXT_OPTIONS)
        page = await context.new_page()
        await page.goto(url, wait_until="load", timeout=timeout_ms)
        await page.wait_for_selector("h1", timeout=30_000)
        html = await page.content()
        await browser.close()
    return BeautifulSoup(html, "html.parser")


def extract_price(soup: BeautifulSoup) -> str | None:
    """Extract price from Harry Winston product page."""
    price_pattern = re.compile(r"[\$¥€£][\d,]+")
    for tag in soup.find_all(string=price_pattern):
        text = tag.strip()
        match = price_pattern.search(text)
        if match and len(text) < 30:
            return match.group()
    full_text = soup.get_text(separator=" ")
    match = price_pattern.search(full_text)
    return match.group() if match else None


def extract_carats(soup: BeautifulSoup) -> tuple[str | None, float | None, list[str]]:
    """Extract per-stone carat details and stone types.

    Returns (formatted_string, total_ct, stones):
        formatted_string : e.g. "サファイア: 1.08ct / ダイヤモンド: 1.19ct"
        total_ct         : sum of all carat values as float
        stones           : deduplicated ordered list, e.g. ["サファイア", "ダイヤモンド"]
    """
    desc_div = soup.find("div", class_="pdp-header__product-description")
    if not desc_div:
        return None, None, []

    text = desc_div.get_text(" ", strip=True)
    details: list[tuple[str | None, float]] = []

    # Japanese: 石名（計約X.XXカラット）
    ja_pat = re.compile(r'([ァ-ヾ・]+)（[^）]*?([\d.]+)カラット）')
    for stone, ct in ja_pat.findall(text):
        details.append((stone, float(ct)))

    if not details:
        # English: "N round/pear Stones (X.XX carats)"
        en_stone_ct = re.compile(
            r'[\d]+\s+[\w\s\-]+?((?:[A-Z][a-z]+(?:\s[a-z]+)?)s?)\s*\('
            r'(?:approximately\s+)?([\d.]+)\s*carats?\)',
            re.I,
        )
        for stone, ct_str in en_stone_ct.findall(text):
            details.append((stone.rstrip('s').strip().title(), float(ct_str)))

    if not details:
        # Fallback: carat numbers only
        for ct_str in re.findall(r'([\d.]+)\s*carats?', text, re.I):
            details.append((None, float(ct_str)))

    if not details:
        return None, None, []

    total_ct = round(sum(ct for _, ct in details), 2)
    formatted = " / ".join(
        f"{stone}: {ct}ct" if stone else f"{ct}ct"
        for stone, ct in details
    )
    seen: set[str] = set()
    stones: list[str] = []
    for stone, _ in details:
        if stone and stone not in seen:
            seen.add(stone)
            stones.append(stone)

    return formatted, total_ct, stones


def extract_product_info(soup: BeautifulSoup) -> dict:
    """Extract product name, price, price_per_ct, stones, carats, SKU, and description."""
    result = {}

    h1 = soup.find("h1")
    result["name"] = h1.get_text(strip=True) if h1 else None
    result["price"] = extract_price(soup)

    # Carat details + stone types
    result["carats"], result["total_ct"], result["stones"] = extract_carats(soup)

    # Price per carat
    price_str = result.get("price")
    total_ct = result.get("total_ct")
    if price_str and total_ct and total_ct > 0:
        price_num = float(re.sub(r"[^\d]", "", price_str))
        result["price_per_ct"] = round(price_num / total_ct)
    else:
        result["price_per_ct"] = None

    # Description — prefer the dedicated PDP div
    desc_div = soup.find("div", class_="pdp-header__product-description")
    if desc_div:
        p_tag = desc_div.find("p")
        result["description"] = p_tag.get_text(strip=True) if p_tag else None
    elif h1:
        for sibling in h1.find_all_next("p"):
            text = sibling.get_text(strip=True)
            if text and len(text) > 20:
                result["description"] = text
                break

    # SKU
    sku_pattern = re.compile(r"[A-Z]{2,}[A-Z0-9]{6,}")
    for tag in soup.find_all(string=sku_pattern):
        text = tag.strip()
        if len(text) < 40:
            match = sku_pattern.search(text)
            if match:
                result["sku"] = match.group()
                break

    return result


In [46]:
soup = await fetch_page(URL)
info = extract_product_info(soup)

print("=" * 50)
print(f"商品名        : {info.get('name')}")
print(f"価格          : {info.get('price')}")
print(f"石の種類      : {', '.join(info.get('stones') or [])}")
print(f"カラット      : {info.get('carats')}")
print(f"合計ct        : {info.get('total_ct')}")
print(f"円/ct         : {info.get('price_per_ct'):,}" if info.get('price_per_ct') else "円/ct         : N/A")
print(f"SKU           : {info.get('sku')}")
print(f"説明          : {info.get('description')}")
print("=" * 50)


商品名        : スパークリングクラスター・サファイア&アクアマリン・リング
価格          : ¥4,059,000
石の種類      : ペアシェイプ・サファイア, ラウンド・アクアマリン, ペアシェイプ・ダイヤモンド
カラット      : ペアシェイプ・サファイア: 1.08ct / ラウンド・アクアマリン: 0.38ct / ペアシェイプ・ダイヤモンド: 1.19ct
合計ct        : 2.65
円/ct         : 1,531,698
SKU           : FRSAQPCLRFSPC
説明          : 2個のラウンド及び1個のペアシェイプ・サファイア（計約1.08カラット）、2個のラウンド・アクアマリン（計約0.38カラット）、7個のラウンド及びペアシェイプ・ダイヤモンド（計約1.19カラット）、プラチナ製。


## カテゴリページから全商品URLを収集 → 一括取得


In [35]:
import json

_BASE = "https://www.harrywinston.com"

_SXA_SEARCH = "https://www.harrywinston.com/ja/sxa/search/results/"


async def fetch_category_urls(category_url: str, max_items: int = 500) -> list[str]:
    """Fetch all product URLs for any Harry Winston category page.

    Intercepts the SXA search API call the category page makes, then re-fires
    it with a large page size to get every product in one request.
    """
    captured_api_url: list[str] = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent=_USER_AGENT,
            locale="ja-JP",
            extra_http_headers={"Accept-Language": "ja,en-US;q=0.9,en;q=0.8"},
        )
        page = await context.new_page()

        def on_request(req):
            if "sxa/search/results" in req.url and not captured_api_url:
                captured_api_url.append(req.url)

        page.on("request", on_request)
        await page.goto(category_url, wait_until="load", timeout=60_000)
        await asyncio.sleep(2)
        await browser.close()

    if not captured_api_url:
        raise RuntimeError(f"SXA search API not found on {category_url}")

    # Replace page size parameter with max_items
    import re as _re
    api_url = _re.sub(r"&p=\d+", f"&p={max_items}", captured_api_url[0])
    if "&p=" not in api_url:
        api_url += f"&p={max_items}"

    # Fetch JSON using Playwright (bypasses HTTPS/h2 issues with plain curl)
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent=_USER_AGENT,
            locale="ja-JP",
            extra_http_headers={
                "Accept-Language": "ja,en-US;q=0.9,en;q=0.8",
                "Referer": category_url,
            },
        )
        page = await context.new_page()
        await page.goto(api_url, wait_until="load", timeout=60_000)
        body = await page.evaluate("document.body.innerText")
        await browser.close()

    data = json.loads(body)
    urls = [_BASE + r["Url"] for r in data["Results"]]
    print(f"カテゴリ: {category_url}")
    print(f"商品数  : {data['Count']} 件取得")
    return urls


# ---- 使用例 ----
CATEGORY_URL = "https://www.harrywinston.com/ja/fine-jewelry/rings"
URLS = await fetch_category_urls(CATEGORY_URL)
for u in URLS:
    print(u)


カテゴリ: https://www.harrywinston.com/ja/fine-jewelry/rings
商品数  : 103 件取得
https://www.harrywinston.com/ja/products/sparkling-cluster-by-harry-winston/sparkling-cluster-pink-sapphire-ruby-and-diamond-ring-frrbpspclrfspc
https://www.harrywinston.com/ja/products/classic-winston/classic-winston-cushion-cut-sapphire-ring-rgspcu100tb
https://www.harrywinston.com/ja/products/diamond-loop-by-harry-winston/diamond-loop-full-motif-diamond-ring-frdprp1ml4c
https://www.harrywinston.com/ja/products/lily-cluster-by-harry-winston/lily-cluster-diamond-ring-large-frdpmqrflc
https://www.harrywinston.com/ja/products/classic-winston/yellow-diamond-and-diamond-ring-rgyedmps090ps-060
https://www.harrywinston.com/ja/products/lily-cluster-by-harry-winston/lily-cluster-diamond-ring-small-frdpsm1mlc
https://www.harrywinston.com/ja/products/lily-cluster-by-harry-winston/infinite-lily-cluster-ring-frdpmqrfilc
https://www.harrywinston.com/ja/products/forget-me-not-by-harry-winston/forget-me-not-pink-sapphire-and-dia

In [ ]:
_CONCURRENCY = 3   # 同時リクエスト数
_OUTPUT_CSV = "/home/kazumasa/projects/notebooks/hw_scraped.csv"

_COLUMNS = ["name", "price", "price_per_ct", "stones", "carats", "total_ct", "sku", "description", "url"]


async def scrape_one(sem: asyncio.Semaphore, context, url: str) -> dict:
    async with sem:
        try:
            page = await context.new_page()
            await page.goto(url, wait_until="load", timeout=60_000)
            await page.wait_for_selector("h1", timeout=30_000)
            soup = BeautifulSoup(await page.content(), "html.parser")
            await page.close()
            info = extract_product_info(soup)
            info["url"] = url
            print(f"[OK] {info.get('name')} | {info.get('price')} | {', '.join(info.get('stones') or [])}")
            return info
        except Exception as e:
            print(f"[ERROR] {url.split('/')[-1]}: {e}")
            return {col: None for col in _COLUMNS} | {"url": url}


async def scrape_all(urls: list[str]) -> pd.DataFrame:
    sem = asyncio.Semaphore(_CONCURRENCY)
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(**_CONTEXT_OPTIONS)
        records = await asyncio.gather(*[scrape_one(sem, context, u) for u in urls])
        await browser.close()
    df = pd.DataFrame(list(records), columns=_COLUMNS)
    df.to_csv(_OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"\n保存完了: {_OUTPUT_CSV}  ({len(df)} 件)")
    return df


df = await scrape_all(URLS)
df
